# Medical Dialogue Data Preprocessing

This notebook handles preprocessing of raw medical dialogue JSON data to prepare it for fine-tuning Mistral 7B. The preprocessing steps include:

1. Loading raw JSON data containing medical dialogues
2. Cleaning and formatting dialogues
3. Combining patient queries and doctor responses
4. Saving processed datasets for training and validation

## Import Required Libraries

In [ ]:
import json
import os
import pandas as pd
import logging
import re
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
import random

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Define Configuration

In [13]:
# Configuration
DATA_DIR = Path("dataset")
PROCESSED_DATA_DIR = Path("processed_data")
TRAIN_PATH = DATA_DIR / "english-train.json"
DEV_PATH = DATA_DIR / "english-dev.json"
TEST_PATH = DATA_DIR / "english-test.json"

# Create output directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(exist_ok=True, parents=True)

# Clean up any existing processed files to avoid confusion
import shutil
if PROCESSED_DATA_DIR.exists():
    for file in PROCESSED_DATA_DIR.glob("*.json"):
        file.unlink()
    logger.info(f"Cleaned up existing processed files in {PROCESSED_DATA_DIR}")

# Output paths
PROCESSED_TRAIN_PATH = PROCESSED_DATA_DIR / "processed_train.json"
PROCESSED_DEV_PATH = PROCESSED_DATA_DIR / "processed_dev.json"
PROCESSED_TEST_PATH = PROCESSED_DATA_DIR / "processed_test.json"

2025-08-28 18:03:29,937 - INFO - Cleaned up existing processed files in processed_data


## Helper Functions for Data Loading and Cleaning

In [ ]:
def load_json_data(file_path):
    """
    Load JSON data from the specified file path with error handling.
    
    Args:
        file_path (Path): Path to the JSON file
        
    Returns:
        list: List of dialogue data or empty list if file not found
    """
    try:
        if not file_path.exists():
            logger.warning(f"File not found: {file_path}")
            return []
        
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        logger.info(f"Successfully loaded {len(data)} dialogues from {file_path}")
        return data
    except json.JSONDecodeError:
        logger.error(f"Error decoding JSON from {file_path}")
        return []
    except Exception as e:
        logger.error(f"Error loading data from {file_path}: {str(e)}")
        return []

def clean_text(text):
    """
    Clean text by removing extra whitespaces
    
    Args:
        text (str): Input text to clean
        
    Returns:
        str: Cleaned text
    """
    if not isinstance(text, str):
        return ""
    
    # Replace multiple whitespaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing whitespace
    text = text.strip()
    return text

def format_dialogue(entry):
    """
    Format a dialogue entry into a standardized format.
    
    Args:
        entry (dict): Dialogue entry with description and utterances fields
        
    Returns:
        dict: Processed entry with combined text field
    """
    try:
        # The dataset format contains 'utterances' as a list where:
        # - utterances[0] typically contains patient's query starting with "patient:"
        # - utterances[1] typically contains doctor's response starting with "doctor:"
        
        utterances = entry.get('utterances', [])
        description = clean_text(entry.get('description', ''))
        
        # Extract patient query and doctor response from utterances
        patient_query = ""
        doctor_response = ""
        
        # Process patient query from utterances
        if len(utterances) > 0:
            patient_text = clean_text(utterances[0])
            if patient_text.lower().startswith("patient:"):
                patient_query = patient_text[patient_text.lower().find(":")+1:].strip()
            else:
                patient_query = patient_text  # Use the whole text if no "patient:" prefix
        
        # Process doctor response from utterances
        if len(utterances) > 1:
            doctor_text = clean_text(utterances[1])
            if doctor_text.lower().startswith("doctor:"):
                doctor_response = doctor_text[doctor_text.lower().find(":")+1:].strip()
            else:
                doctor_response = doctor_text  # Use the whole text if no "doctor:" prefix
        
        # If no patient query found in utterances, use description as fallback
        if not patient_query and description:
            patient_query = description
            
        # Handle missing fields
        if not patient_query:
            logger.warning("Missing patient query in entry")
            patient_query = "[No patient query provided]"
        
        if not doctor_response:
            logger.warning("Missing doctor response in entry")
            doctor_response = "[No doctor response provided]"
            
        # Combine into a single text field
        combined_text = f"Patient: {patient_query}\nDoctor: {doctor_response}"
        
        # Return processed entry
        return {
            "text": combined_text,
            "patient_query": patient_query,
            "doctor_response": doctor_response,
            "description": description
        }
    except Exception as e:
        logger.error(f"Error formatting dialogue: {str(e)}")
        return {"text": "", "patient_query": "", "doctor_response": "", "description": ""}

## Process Datasets

In [ ]:
def process_dataset(input_path, output_path):
    """
    Process a dataset from input path and save to output path.
    
    Args:
        input_path (Path): Path to raw data file
        output_path (Path): Path to save processed data
        
    Returns:
        int: Number of successfully processed entries
    """
    # Load raw data from using load_json_data
    raw_data = load_json_data(input_path)

    if not raw_data:
        logger.warning(f"No data to process from {input_path}")
        return 0
    
    # Process each entry
    logger.info(f"Processing {len(raw_data)} dialogues from {input_path}")
    processed_data = []
    
    for entry in tqdm(raw_data, desc="Processing dialogues"):
        processed_entry = format_dialogue(entry)
        # Only add entries that have non-empty text
        if processed_entry["text"].strip():
            processed_data.append(processed_entry)
    
    # Save processed data
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, ensure_ascii=False, indent=2) # save processed data
        logger.info(f"Successfully saved {len(processed_data)} processed dialogues to {output_path}")
    except Exception as e:
        logger.error(f"Error saving processed data to {output_path}: {str(e)}")
        
    return len(processed_data)

## Process Training Dataset

In [16]:
# Process training data
train_count = process_dataset(TRAIN_PATH, PROCESSED_TRAIN_PATH)
print(f"Processed {train_count} training examples")

2025-08-28 18:03:30,132 - INFO - Successfully loaded 482 dialogues from dataset/english-train.json
2025-08-28 18:03:30,132 - INFO - Processing 482 dialogues from dataset/english-train.json
2025-08-28 18:03:30,132 - INFO - Processing 482 dialogues from dataset/english-train.json


Processing dialogues:   0%|          | 0/482 [00:00<?, ?it/s]

2025-08-28 18:03:30,163 - INFO - Successfully saved 482 processed dialogues to processed_data/processed_train.json


Processed 482 training examples


## Process Development Dataset

In [17]:
# Process dev data
dev_count = process_dataset(DEV_PATH, PROCESSED_DEV_PATH)
print(f"Processed {dev_count} development examples")

2025-08-28 18:03:30,175 - INFO - Successfully loaded 60 dialogues from dataset/english-dev.json
2025-08-28 18:03:30,175 - INFO - Processing 60 dialogues from dataset/english-dev.json
2025-08-28 18:03:30,175 - INFO - Processing 60 dialogues from dataset/english-dev.json


Processing dialogues:   0%|          | 0/60 [00:00<?, ?it/s]

2025-08-28 18:03:30,183 - INFO - Successfully saved 60 processed dialogues to processed_data/processed_dev.json


Processed 60 development examples


## Process Test Dataset 

In [18]:
# Process test data
test_count = process_dataset(TEST_PATH, PROCESSED_TEST_PATH)
print(f"Processed {test_count} test examples")

2025-08-28 18:03:30,200 - INFO - Successfully loaded 61 dialogues from dataset/english-test.json
2025-08-28 18:03:30,200 - INFO - Processing 61 dialogues from dataset/english-test.json
2025-08-28 18:03:30,200 - INFO - Processing 61 dialogues from dataset/english-test.json


Processing dialogues:   0%|          | 0/61 [00:00<?, ?it/s]

2025-08-28 18:03:30,207 - INFO - Successfully saved 61 processed dialogues to processed_data/processed_test.json


Processed 61 test examples


## Verify Processed Data

In [19]:
def display_sample_data(file_path, num_samples=5):
    """
    Display sample data from a processed file.
    
    Args:
        file_path (Path): Path to processed JSON file
        num_samples (int): Number of samples to display
    """
    try:
        if not file_path.exists():
            print(f"File {file_path} not found")
            return
            
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        print(f"\n=== Sample data from {file_path} (showing {min(num_samples, len(data))} of {len(data)} entries) ===")
        
        samples = random.sample(data, min(num_samples, len(data)))
        for i, sample in enumerate(samples, 1):
            print(f"\nSample {i}:")
            print(f"Patient query: {sample['patient_query']}")
            print(f"Doctor response: {sample['doctor_response']}")
            print(f"Combined text:\n{sample['text']}")
            print("---")
    except Exception as e:
        print(f"Error displaying sample data: {str(e)}")

# Display sample data from each processed file
display_sample_data(PROCESSED_TRAIN_PATH)
display_sample_data(PROCESSED_DEV_PATH)


=== Sample data from processed_data/processed_train.json (showing 5 of 482 entries) ===

Sample 1:
Patient query: i have had constant chest inf was hospitalised for 4 days said they think phnewmonia out now still on lots of anti biotics and steroids however i'm still sweating have right lower back pain and still really coughing and short of breath. i am mbl deficency and alpha 1
Doctor response: helloyes according to the history it might be pneumonia.continue the current treatment.regardsdr.jolanda
Combined text:
Patient: i have had constant chest inf was hospitalised for 4 days said they think phnewmonia out now still on lots of anti biotics and steroids however i'm still sweating have right lower back pain and still really coughing and short of breath. i am mbl deficency and alpha 1
Doctor: helloyes according to the history it might be pneumonia.continue the current treatment.regardsdr.jolanda
---

Sample 2:
Patient query: good day my 2 year old toddler has been coughing and now she